# Capstone: The Quality Inspector — Review Agent

## Module 7 — Capstone Project (Notebook 5 of 6)

The Review Agent is Castalia Scholar's **quality gate**. It evaluates research reports produced by the Writing Agent against multi-dimensional rubrics, identifies specific issues, generates actionable feedback, and decides whether a report is ready to publish or needs revision.

Without a review step, generated reports often contain subtle factual errors, missing coverage, weak citations, and structural problems. The Review Agent catches these systematically.

### What You'll Build

1. **ReviewRubric** — a multi-dimensional quality evaluation framework (accuracy, completeness, clarity, citation quality, coherence)
2. **DimensionReviewer** — specialized reviewers for each quality dimension
3. **FeedbackGenerator** — converts review scores into prioritized, actionable feedback items
4. **ReviewDecision** — pass/revise decision logic with weighted scoring
5. **ReviewAgent** — the complete review pipeline using reflection patterns from Notebook 07
6. **Tests with flawed reports** — verify reviewers catch factual errors, gaps, and missing citations
7. **Review loop preview** — how reviewer feedback flows back to the writer

### Learning Objectives

By the end of this notebook you will:
- Build a **multi-dimensional quality review** system with weighted rubrics
- Implement **factual verification** by cross-referencing reports against source documents
- Generate **actionable, prioritized feedback** with specific locations and suggested fixes
- Make **data-driven pass/revise decisions** based on weighted quality scores
- Understand how review quality scales from single-pass to multi-dimensional evaluation

---

> **Prerequisites**: Notebooks 07 (reflection), 32–35 (capstone architecture, retrieval, analysis, writing agents).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

Same Qwen3-14B setup. The model will act as a quality reviewer — evaluating reports across multiple dimensions and generating structured feedback.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## Part 1: The Review Agent's Role in Castalia Scholar

### Where the Review Agent Fits

```
User Question
     │
     ▼
┌─────────────┐
│   Planner   │  ── decomposes question
└──────┬──────┘
       ▼
┌─────────────┐
│  Retriever  │  ── finds relevant sources
└──────┬──────┘
       ▼
┌─────────────┐
│  Analyzer   │  ── synthesizes findings
└──────┬──────┘
       ▼
┌─────────────┐
│   Writer    │  ── produces report
└──────┬──────┘
       ▼
┌══════════════════════════════════════════════════════════════╗
║                   REVIEW AGENT  ◄── YOU ARE HERE            ║
║                                                              ║
║  1. Evaluate report across 5 quality dimensions              ║
║  2. Generate actionable feedback with locations              ║
║  3. Decide: PASS → deliver  or  REVISE → send back to Writer║
╚══════════════════════════════════════════════════════════════╝
       │
       ├──── PASS ──► Final Report to User
       │
       └──── REVISE ──► Back to Writer with feedback
                        (max 2 revision rounds)
```

The Review Agent is the **last line of defense** before the user sees the report. Without it, errors slip through. With it, quality improves measurably with each revision round.

### What the Review Agent Evaluates

| Dimension | What It Checks | Why It Matters |
|-----------|---------------|----------------|
| **Accuracy** | Facts match source documents | Prevents hallucination propagation |
| **Completeness** | All sub-questions addressed | Ensures thorough coverage |
| **Clarity** | Writing is clear and readable | Usability for the end user |
| **Citation Quality** | Claims backed by sources | Trustworthiness and verifiability |
| **Coherence** | Logical flow between sections | Professional, polished output |

## Part 2: Quality Rubrics — The Evaluation Framework

A rubric turns subjective "is this good?" into **measurable, repeatable evaluation**. Each dimension has explicit scoring criteria so the LLM knows exactly what to look for.

In [ ]:
@dataclass
class ReviewRubric:
    """A single dimension of the quality evaluation framework."""
    dimension: str          # e.g., "accuracy", "completeness"
    weight: float           # relative importance (higher = more important)
    scoring_criteria: str   # detailed description of what each score level means
    max_score: int = 10     # score out of this maximum

    def to_prompt_section(self) -> str:
        """Format this rubric dimension for inclusion in a review prompt."""
        return (
            f"### {self.dimension.upper()} (weight: {self.weight}, max: {self.max_score})\n"
            f"{self.scoring_criteria}\n"
            f"Score guide: 1-3 = Poor, 4-5 = Below Average, 6-7 = Adequate, 8-9 = Good, 10 = Excellent"
        )

# ── Define the 5 quality dimensions ──

RUBRIC_DIMENSIONS = [
    ReviewRubric(
        dimension="accuracy",
        weight=2.5,
        scoring_criteria=(
            "Are all factual claims in the report supported by the source documents? "
            "Check for: (1) statements that contradict sources, (2) fabricated facts not in any source, "
            "(3) misrepresented statistics or findings, (4) correct attribution of claims to sources."
        ),
    ),
    ReviewRubric(
        dimension="completeness",
        weight=2.0,
        scoring_criteria=(
            "Does the report address all aspects of the original research question? "
            "Check for: (1) all sub-questions answered, (2) key topics from sources covered, "
            "(3) no significant gaps in coverage, (4) appropriate depth on each topic."
        ),
    ),
    ReviewRubric(
        dimension="clarity",
        weight=1.5,
        scoring_criteria=(
            "Is the report clear, well-written, and accessible to the target audience? "
            "Check for: (1) jargon explained on first use, (2) sentences are concise, "
            "(3) paragraphs have clear topic sentences, (4) no ambiguous pronouns or vague references."
        ),
    ),
    ReviewRubric(
        dimension="citation_quality",
        weight=2.0,
        scoring_criteria=(
            "Are claims properly backed by citations to source documents? "
            "Check for: (1) key claims have source references, (2) citations match the actual source content, "
            "(3) no orphan citations (cited but not relevant), (4) citation format is consistent."
        ),
    ),
    ReviewRubric(
        dimension="coherence",
        weight=1.5,
        scoring_criteria=(
            "Does the report flow logically from section to section? "
            "Check for: (1) smooth transitions between paragraphs, (2) ideas build on each other, "
            "(3) no contradictions within the report itself, (4) conclusion follows from the body."
        ),
    ),
]

# Display the rubric
total_weight = sum(r.weight for r in RUBRIC_DIMENSIONS)
print("Quality Evaluation Rubric")
print("=" * 60)
for r in RUBRIC_DIMENSIONS:
    pct = r.weight / total_weight * 100
    print(f"  {r.dimension:<20s} weight={r.weight}  ({pct:.0f}% of total)")
print(f"{'':>20s} total={total_weight}")
print(f"\nMax possible score: {RUBRIC_DIMENSIONS[0].max_score}/10 per dimension")

In [ ]:
def build_review_prompt(rubric_dims: List[ReviewRubric]) -> str:
    """Build a complete review prompt from all rubric dimensions."""
    sections = []
    for r in rubric_dims:
        sections.append(r.to_prompt_section())

    return (
        "You are a rigorous research report reviewer. Evaluate the report below "
        "across the following quality dimensions.\n\n"
        + "\n\n".join(sections)
        + "\n\nFor EACH dimension, respond with a JSON object:\n"
        + '{"reviews": [{"dimension": "...", "score": N, "issues": ["issue1", ...], '
        + '"strengths": ["strength1", ...], "suggestions": ["fix1", ...]}], '
        + '"overall_score": N, "decision": "PASS or REVISE", "summary": "..."}'
    )

review_prompt = build_review_prompt(RUBRIC_DIMENSIONS)
print("Review prompt preview (first 600 chars):")
print(review_prompt[:600] + "...")

## Part 3: DimensionReviewer — Specialized Evaluation

Rather than asking the LLM to evaluate all dimensions at once (which often leads to shallow reviews), we build **specialized reviewers** for each dimension. Each reviewer gets a focused prompt that deeply evaluates just one aspect.

This mirrors how real peer review works — a statistics reviewer checks the numbers, a domain expert checks the content, an editor checks the writing.

In [ ]:
@dataclass
class DimensionScore:
    """Result of reviewing one dimension."""
    dimension: str
    score: int              # 1-10
    issues: List[str]       # specific problems found
    strengths: List[str]    # what's done well
    suggestions: List[str]  # actionable fixes

    def summary(self) -> str:
        lines = [f"{self.dimension.upper()}: {self.score}/10"]
        if self.strengths:
            lines.append(f"  ✓ Strengths: {'; '.join(self.strengths[:2])}")
        if self.issues:
            lines.append(f"  ✗ Issues: {'; '.join(self.issues[:2])}")
        if self.suggestions:
            lines.append(f"  → Fix: {'; '.join(self.suggestions[:2])}")
        return "\n".join(lines)


class DimensionReviewer:
    """Evaluates a report on a single quality dimension."""

    def __init__(self, rubric: ReviewRubric):
        self.rubric = rubric

    def review(self, report: str, sources: List[str] = None,
               original_question: str = None) -> DimensionScore:
        """Review the report on this dimension. Returns a DimensionScore."""
        system_prompt = (
            f"You are a specialist reviewer focused ONLY on {self.rubric.dimension}.\n"
            f"{self.rubric.scoring_criteria}\n\n"
            f"Score from 1 (terrible) to 10 (flawless).\n"
            f"Respond ONLY with JSON: "
            f'{{"score": N, "issues": [...], "strengths": [...], "suggestions": [...]}}'
        )

        user_parts = [f"REPORT TO REVIEW:\n{report}"]
        if sources:
            src_text = "\n---\n".join(sources[:5])
            user_parts.append(f"\nSOURCE DOCUMENTS:\n{src_text}")
        if original_question:
            user_parts.append(f"\nORIGINAL QUESTION: {original_question}")

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "\n".join(user_parts)},
        ]

        raw = generate(messages, max_new_tokens=400, temperature=0.3, do_sample=True)
        return self._parse(raw)

    def _parse(self, raw: str) -> DimensionScore:
        """Parse LLM output into a DimensionScore."""
        try:
            match = re.search(r'\{[\s\S]*\}', raw)
            if match:
                data = json.loads(match.group())
                return DimensionScore(
                    dimension=self.rubric.dimension,
                    score=min(10, max(1, int(data.get("score", 5)))),
                    issues=data.get("issues", []),
                    strengths=data.get("strengths", []),
                    suggestions=data.get("suggestions", []),
                )
        except (json.JSONDecodeError, ValueError):
            pass
        # Fallback
        return DimensionScore(
            dimension=self.rubric.dimension, score=5,
            issues=["Could not parse review output"],
            strengths=[], suggestions=["Re-run review"],
        )

# Build a reviewer for each dimension
dimension_reviewers = {r.dimension: DimensionReviewer(r) for r in RUBRIC_DIMENSIONS}
print(f"✓ Built {len(dimension_reviewers)} dimension reviewers:")
for name in dimension_reviewers:
    print(f"  • {name}")

### 3.1 — Test a Single Dimension Reviewer

Let's create a short sample report and test the accuracy reviewer on it. We deliberately include a factual error to verify the reviewer catches it.

In [ ]:
# A sample report with a deliberate factual error
sample_report = """
# Neural Network Training Methods

Neural networks are trained using backpropagation, which was first described by
Geoffrey Hinton in 1986. The algorithm computes gradients of the loss function
with respect to each weight using the chain rule of calculus.

Modern training typically uses stochastic gradient descent (SGD) or its variants
like Adam optimizer. Adam was introduced by Kingma and Ba in 2014 and combines
momentum with RMSProp. A key advantage of Adam is that it requires no
learning rate tuning at all, making it completely plug-and-play.

Batch normalization, introduced by Ioffe and Szegedy in 2015, normalizes
activations within each layer, which accelerates training convergence.
"""

# Source documents the report was based on
sample_sources = [
    "Backpropagation was popularized by Rumelhart, Hinton, and Williams in 1986, "
    "though earlier versions existed. It computes gradients using the chain rule.",

    "Adam optimizer (Kingma & Ba, 2014) combines momentum and adaptive learning rates. "
    "While it converges faster than vanilla SGD, it still requires learning rate selection "
    "and careful tuning of beta parameters for best results.",

    "Batch normalization (Ioffe & Szegedy, 2015) normalizes layer inputs during training, "
    "reducing internal covariate shift and allowing higher learning rates.",
]

# Test the accuracy reviewer
accuracy_reviewer = dimension_reviewers["accuracy"]
accuracy_score = accuracy_reviewer.review(
    report=sample_report,
    sources=sample_sources,
    original_question="How are neural networks trained?"
)

print("ACCURACY REVIEW RESULT:")
print(accuracy_score.summary())
print(f"\n(The error: report claims Adam needs 'no learning rate tuning at all' — sources say otherwise)")

## Part 4: FeedbackGenerator — Actionable Feedback

Raw scores aren't enough. The writer needs to know **exactly what to fix and where**. The FeedbackGenerator converts dimension scores into prioritized, actionable feedback items.

Each feedback item specifies:
- **Where** in the report the issue is (section/paragraph)
- **What** is wrong (issue type and description)
- **How** to fix it (concrete suggestion)
- **How urgent** (severity: critical, major, minor)

In [ ]:
@dataclass
class FeedbackItem:
    """A single piece of actionable feedback for the writer."""
    section: str        # where in the report
    issue_type: str     # accuracy, completeness, clarity, citation, coherence
    severity: str       # critical, major, minor
    description: str    # what's wrong
    suggestion: str     # how to fix it

    def __str__(self):
        icon = {"critical": "🔴", "major": "🟡", "minor": "🟢"}.get(self.severity, "⚪")
        return f"{icon} [{self.severity.upper()}] ({self.issue_type}) {self.section}: {self.description}\n   → Fix: {self.suggestion}"


class FeedbackGenerator:
    """Converts review scores into prioritized, actionable feedback."""

    SEVERITY_MAP = {
        (1, 3): "critical",   # scores 1-3: critical issues
        (4, 5): "major",      # scores 4-5: major issues
        (6, 7): "minor",      # scores 6-7: minor issues
    }

    def generate_feedback(self, dimension_scores: List[DimensionScore],
                          report: str) -> List[FeedbackItem]:
        """Generate feedback items from dimension scores."""
        feedback_items = []

        for ds in dimension_scores:
            severity = self._score_to_severity(ds.score)
            if severity is None:
                continue  # score 8+ means no feedback needed

            # Convert each issue + suggestion pair into a FeedbackItem
            for i, issue in enumerate(ds.issues):
                suggestion = ds.suggestions[i] if i < len(ds.suggestions) else "Address this issue."
                section = self._locate_issue(issue, report)
                feedback_items.append(FeedbackItem(
                    section=section,
                    issue_type=ds.dimension,
                    severity=severity,
                    description=issue,
                    suggestion=suggestion,
                ))

        # Sort by severity: critical first, then major, then minor
        severity_order = {"critical": 0, "major": 1, "minor": 2}
        feedback_items.sort(key=lambda f: severity_order.get(f.severity, 3))
        return feedback_items

    def _score_to_severity(self, score: int) -> Optional[str]:
        """Map a score to a severity level. Returns None if score is high enough."""
        for (low, high), severity in self.SEVERITY_MAP.items():
            if low <= score <= high:
                return severity
        return None  # score 8+, no issues worth reporting

    def _locate_issue(self, issue: str, report: str) -> str:
        """Try to locate which section of the report an issue refers to."""
        # Simple heuristic: find section headers and match keywords
        sections = re.findall(r'^#+\s+(.+)', report, re.MULTILINE)
        issue_lower = issue.lower()
        for section in sections:
            # Check if any section keywords appear in the issue
            words = section.lower().split()
            if any(w in issue_lower for w in words if len(w) > 3):
                return section
        return "General"

    def format_for_writer(self, feedback_items: List[FeedbackItem]) -> str:
        """Format all feedback into a prompt section for the writer agent."""
        if not feedback_items:
            return "No issues found. Report quality is excellent."
        lines = ["REVISION FEEDBACK (ordered by priority):", ""]
        for i, item in enumerate(feedback_items, 1):
            lines.append(f"{i}. {item}")
            lines.append("")
        counts = defaultdict(int)
        for item in feedback_items:
            counts[item.severity] += 1
        lines.append(f"Summary: {counts.get('critical', 0)} critical, "
                      f"{counts.get('major', 0)} major, {counts.get('minor', 0)} minor issues")
        return "\n".join(lines)


feedback_gen = FeedbackGenerator()
print("✓ FeedbackGenerator ready")
print(f"  Severity mapping: {FeedbackGenerator.SEVERITY_MAP}")

In [ ]:
# Generate feedback from our earlier accuracy review
test_scores = [accuracy_score]
feedback_items = feedback_gen.generate_feedback(test_scores, sample_report)

print("GENERATED FEEDBACK:")
print("=" * 60)
for item in feedback_items:
    print(item)
    print()

print("\nFORMATTED FOR WRITER:")
print("-" * 60)
print(feedback_gen.format_for_writer(feedback_items))

## Part 5: ReviewDecision — Pass or Revise?

Not every report needs revision. The ReviewDecision class computes a **weighted overall score** and applies a threshold to decide.

The key insight: dimensions have different weights. An accuracy score of 4/10 is far worse than a clarity score of 4/10, because accuracy has weight 2.5 vs clarity's 1.5.

In [ ]:
@dataclass
class ReviewResult:
    """Complete result of a multi-dimensional review."""
    dimension_scores: List[DimensionScore]
    feedback_items: List[FeedbackItem]
    weighted_score: float
    decision: str           # "PASS" or "REVISE"
    dimensions_to_improve: List[str]
    summary: str

    def display(self):
        """Pretty-print the review result."""
        print(f"{'=' * 60}")
        print(f"REVIEW DECISION: {self.decision}")
        print(f"Overall Weighted Score: {self.weighted_score:.1f}/10.0")
        print(f"{'=' * 60}")
        print()
        for ds in self.dimension_scores:
            bar = "█" * ds.score + "░" * (10 - ds.score)
            print(f"  {ds.dimension:<20s} [{bar}] {ds.score}/10")
        print()
        if self.dimensions_to_improve:
            print(f"Dimensions needing improvement: {', '.join(self.dimensions_to_improve)}")
        print(f"\nFeedback items: {len(self.feedback_items)}")
        if self.feedback_items:
            for item in self.feedback_items[:5]:
                print(f"  {item}")
        print(f"\nSummary: {self.summary}")


class ReviewDecision:
    """Decides whether a report passes review or needs revision."""

    def __init__(self, rubrics: List[ReviewRubric], pass_threshold: float = 7.0,
                 dimension_min: float = 4.0):
        self.rubrics = rubrics
        self.pass_threshold = pass_threshold
        self.dimension_min = dimension_min  # minimum acceptable score per dimension
        self.rubric_map = {r.dimension: r for r in rubrics}

    def decide(self, dimension_scores: List[DimensionScore],
               feedback_items: List[FeedbackItem]) -> ReviewResult:
        """Make a pass/revise decision based on scores."""
        # Compute weighted overall score
        weighted_sum = 0.0
        total_weight = 0.0
        dims_to_improve = []

        for ds in dimension_scores:
            rubric = self.rubric_map.get(ds.dimension)
            if rubric:
                weighted_sum += ds.score * rubric.weight
                total_weight += rubric.weight
                if ds.score < self.dimension_min:
                    dims_to_improve.append(ds.dimension)

        weighted_score = weighted_sum / total_weight if total_weight > 0 else 0.0

        # Decision logic
        if weighted_score >= self.pass_threshold and not dims_to_improve:
            decision = "PASS"
            summary = (f"Report meets quality standards with weighted score "
                       f"{weighted_score:.1f}/{self.pass_threshold}.")
        elif dims_to_improve:
            decision = "REVISE"
            summary = (f"Report scored {weighted_score:.1f}/10 but dimensions "
                       f"{', '.join(dims_to_improve)} scored below minimum {self.dimension_min}.")
        else:
            decision = "REVISE"
            summary = (f"Report scored {weighted_score:.1f}/10, below threshold "
                       f"{self.pass_threshold}/10. Revision needed.")

        return ReviewResult(
            dimension_scores=dimension_scores,
            feedback_items=feedback_items,
            weighted_score=weighted_score,
            decision=decision,
            dimensions_to_improve=dims_to_improve,
            summary=summary,
        )


review_decision = ReviewDecision(RUBRIC_DIMENSIONS, pass_threshold=7.0, dimension_min=4.0)

# Quick demo with a fabricated set of scores
demo_scores = [
    DimensionScore("accuracy", 8, ["Minor date error"], ["Good sourcing"], ["Fix the date"]),
    DimensionScore("completeness", 7, [], ["Covers all topics"], []),
    DimensionScore("clarity", 9, [], ["Very clear writing"], []),
    DimensionScore("citation_quality", 6, ["Missing citation in para 3"], ["Consistent format"], ["Add citation"]),
    DimensionScore("coherence", 8, [], ["Logical flow"], []),
]
demo_feedback = feedback_gen.generate_feedback(demo_scores, "")
demo_result = review_decision.decide(demo_scores, demo_feedback)
demo_result.display()

### 5.1 — How Thresholds Affect Decisions

Let's see how the pass threshold changes outcomes for the same set of scores.

In [ ]:
# Compare different thresholds on the same scores
print("Effect of threshold on pass/revise decision:")
print(f"{'Threshold':<12s} {'Decision':<10s} {'Weighted Score':<16s} {'Dims to Fix'}")
print("-" * 60)

for threshold in [5.0, 6.0, 7.0, 7.5, 8.0, 9.0]:
    rd = ReviewDecision(RUBRIC_DIMENSIONS, pass_threshold=threshold, dimension_min=4.0)
    result = rd.decide(demo_scores, demo_feedback)
    dims = ", ".join(result.dimensions_to_improve) if result.dimensions_to_improve else "none"
    print(f"{threshold:<12.1f} {result.decision:<10s} {result.weighted_score:<16.2f} {dims}")

print(f"\nNote: Even with a low threshold, a single dimension below dimension_min=4.0")
print(f"would force REVISE regardless of the overall score.")

## Part 6: The Complete Review Agent

Now we combine everything into a single `ReviewAgent` class that:
1. Takes a report, source documents, and the original question
2. Evaluates all 5 dimensions using specialized reviewers
3. Generates prioritized feedback
4. Makes a pass/revise decision
5. Uses the **reflection pattern** from Notebook 07 — the reviewer reflects on its own assessment quality

In [ ]:
class ReviewAgent:
    """Complete review agent that evaluates reports across multiple dimensions."""

    def __init__(self, rubrics: List[ReviewRubric] = None,
                 pass_threshold: float = 7.0, dimension_min: float = 4.0):
        self.rubrics = rubrics or RUBRIC_DIMENSIONS
        self.reviewers = {r.dimension: DimensionReviewer(r) for r in self.rubrics}
        self.feedback_gen = FeedbackGenerator()
        self.decision_maker = ReviewDecision(self.rubrics, pass_threshold, dimension_min)
        self.review_history = []

    def review(self, report: str, sources: List[str] = None,
               original_question: str = None, verbose: bool = True) -> ReviewResult:
        """Run a complete multi-dimensional review of the report."""
        if verbose:
            print(f"{'=' * 60}")
            print("REVIEW AGENT — Multi-Dimensional Quality Evaluation")
            print(f"{'=' * 60}")
            print(f"Report length: {len(report)} chars")
            print(f"Sources provided: {len(sources) if sources else 0}")
            print(f"Dimensions: {len(self.rubrics)}")
            print()

        # Step 1: Evaluate each dimension
        dim_scores = []
        for dim_name, reviewer in self.reviewers.items():
            if verbose:
                print(f"  Reviewing {dim_name}...", end=" ")
            t0 = time.time()
            score = reviewer.review(report, sources, original_question)
            elapsed = time.time() - t0
            dim_scores.append(score)
            if verbose:
                print(f"score={score.score}/10  ({elapsed:.1f}s)")

        # Step 2: Generate feedback
        feedback_items = self.feedback_gen.generate_feedback(dim_scores, report)

        # Step 3: Self-reflection — does the review itself make sense?
        dim_scores = self._reflect_on_review(dim_scores, report, verbose)

        # Step 4: Make decision
        result = self.decision_maker.decide(dim_scores, feedback_items)
        self.review_history.append(result)

        if verbose:
            print()
            result.display()

        return result

    def _reflect_on_review(self, dim_scores: List[DimensionScore],
                           report: str, verbose: bool) -> List[DimensionScore]:
        """Reflection step: check if our own review is internally consistent."""
        # Quick sanity check: are scores wildly different from what the report suggests?
        scores_text = ", ".join(f"{ds.dimension}={ds.score}" for ds in dim_scores)
        messages = [
            {"role": "system", "content":
                "You reviewed a report and assigned these dimension scores. "
                "Check if the scores are internally consistent. For example, "
                "if accuracy is high but citation_quality is very low, that might "
                "indicate the accuracy score should be re-examined (accurate claims "
                "need good citations). Respond with JSON: "
                '{"consistent": true/false, "adjustments": [{"dimension": "...", '
                '"original": N, "adjusted": N, "reason": "..."}]}'},
            {"role": "user", "content":
                f"Scores: {scores_text}\n\nReport (first 500 chars): {report[:500]}"},
        ]
        raw = generate(messages, max_new_tokens=300, temperature=0.3, do_sample=True)

        try:
            match = re.search(r'\{[\s\S]*\}', raw)
            if match:
                data = json.loads(match.group())
                adjustments = data.get("adjustments", [])
                if adjustments and verbose:
                    print("\n  [Reflection] Score adjustments:")
                for adj in adjustments:
                    dim = adj.get("dimension", "")
                    new_score = adj.get("adjusted", None)
                    reason = adj.get("reason", "")
                    if new_score is not None:
                        for ds in dim_scores:
                            if ds.dimension == dim:
                                old = ds.score
                                ds.score = min(10, max(1, int(new_score)))
                                if verbose:
                                    print(f"    {dim}: {old} → {ds.score} ({reason})")
        except (json.JSONDecodeError, ValueError):
            pass  # Keep original scores if reflection fails

        return dim_scores

print("✓ ReviewAgent defined")

## Part 7: Testing with Deliberately Flawed Reports

The best way to validate a reviewer is to give it reports with **known defects** and check that it catches them. We build three test cases, each targeting a different dimension.

### 7.1 — Report with Factual Errors

This report contains claims that contradict the source documents. The accuracy reviewer should catch them.

In [ ]:
# ── Test 1: Factual errors ──
flawed_report_accuracy = """
# The Transformer Architecture

The Transformer architecture was introduced by Vaswani et al. in their 2017 paper
"Attention Is All You Need." It replaced recurrent neural networks with a purely
attention-based mechanism.

The key innovation is self-attention, which allows each token to attend to every other
token in the sequence. The original Transformer used 4 attention heads in each layer,
with a model dimension of 512.

Training uses the Adam optimizer with a fixed learning rate of 0.001 throughout
the entire training process. The model was trained on the WMT English-to-French
translation task, achieving a BLEU score of 41.0.

The Transformer has become the foundation for models like BERT, GPT, and T5.
"""

accuracy_sources = [
    "Vaswani et al. (2017) 'Attention Is All You Need' introduced the Transformer. "
    "The original model used 8 attention heads (not 4) with model dimension 512.",

    "The Transformer training used Adam optimizer with a warmup learning rate schedule, "
    "not a fixed rate. The learning rate increases linearly for warmup_steps then decreases "
    "proportionally to the inverse square root of the step number.",

    "On WMT 2014 English-to-French, the Transformer achieved BLEU 41.0, setting a new "
    "state-of-the-art. The big model had 6 layers, 8 heads, and 512-dim embeddings.",
]

review_agent = ReviewAgent(pass_threshold=7.0)
result_accuracy = review_agent.review(
    report=flawed_report_accuracy,
    sources=accuracy_sources,
    original_question="Explain the Transformer architecture",
)
print(f"\nExpected: REVISE (report says 4 heads instead of 8, fixed LR instead of warmup)")

### 7.2 — Report Missing Key Topics

This report only covers part of the original question. The completeness reviewer should identify the gaps.

In [ ]:
# ── Test 2: Missing key topics ──
flawed_report_completeness = """
# Approaches to Training Large Language Models

Large language models (LLMs) are typically trained using next-token prediction on
large text corpora. The model learns to predict the next word given all previous
words. This is called causal language modeling.

The training data usually comes from web crawls, books, and code repositories.
Popular datasets include The Pile, C4, and RedPajama. Data quality is crucial —
deduplication, filtering, and cleaning significantly affect model performance.
"""

completeness_sources = [
    "LLM training involves three main phases: (1) pre-training on large corpora using "
    "next-token prediction, (2) supervised fine-tuning (SFT) on instruction-response pairs, "
    "and (3) reinforcement learning from human feedback (RLHF) for alignment.",

    "Scaling laws (Kaplan et al., 2020) show that model performance improves predictably "
    "with increases in model size, dataset size, and compute budget. Chinchilla (Hoffmann "
    "et al., 2022) showed many models were undertrained relative to their size.",

    "Distributed training techniques like data parallelism, tensor parallelism, and "
    "pipeline parallelism are essential for training models with billions of parameters "
    "across multiple GPUs.",
]

result_completeness = review_agent.review(
    report=flawed_report_completeness,
    sources=completeness_sources,
    original_question="What are the main approaches to training large language models?",
)
print(f"\nExpected: REVISE (missing RLHF/SFT, scaling laws, distributed training)")

### 7.3 — Report with No Citations

This report makes factual claims but never references where the information came from.

In [ ]:
# ── Test 3: No citations ──
flawed_report_citations = """
# Generative Adversarial Networks

GANs consist of two neural networks: a generator and a discriminator. The generator
creates synthetic data, while the discriminator tries to distinguish real from fake.
They are trained adversarially — the generator improves by fooling the discriminator,
and the discriminator improves by catching fakes.

Training GANs is notoriously difficult. Mode collapse occurs when the generator
produces only a narrow range of outputs. Training instability can cause the loss
to oscillate wildly. Various techniques like Wasserstein loss, spectral normalization,
and progressive growing have been proposed to address these issues.

GANs have been applied to image generation, style transfer, super-resolution,
and data augmentation. Recent work on diffusion models has largely superseded GANs
for image generation tasks.
"""

citation_sources = [
    "[Source 1] Goodfellow et al. (2014) introduced GANs with the original adversarial "
    "training framework using a minimax game between generator and discriminator.",

    "[Source 2] Arjovsky et al. (2017) proposed Wasserstein GAN (WGAN) using Earth Mover "
    "distance to improve training stability.",

    "[Source 3] Karras et al. (2018) introduced progressive growing of GANs, training "
    "at increasing resolutions for high-quality image synthesis.",

    "[Source 4] Ho et al. (2020) introduced denoising diffusion probabilistic models "
    "(DDPMs), which have largely replaced GANs for image generation.",
]

result_citations = review_agent.review(
    report=flawed_report_citations,
    sources=citation_sources,
    original_question="Explain generative adversarial networks and their training challenges",
)
print(f"\nExpected: REVISE (no citations in report despite 4 sources available)")

### 7.4 — Test Results Summary

Let's see how the reviewer performed across all three deliberately flawed reports.

In [ ]:
# Summarize all test results
print("REVIEW AGENT TEST RESULTS")
print("=" * 70)
print(f"{'Test Case':<30s} {'Decision':<10s} {'Score':<8s} {'Dims to Fix'}")
print("-" * 70)

test_results = [
    ("Factual errors", result_accuracy),
    ("Missing topics", result_completeness),
    ("No citations", result_citations),
]

for name, result in test_results:
    dims = ", ".join(result.dimensions_to_improve) if result.dimensions_to_improve else "none"
    print(f"{name:<30s} {result.decision:<10s} {result.weighted_score:<8.1f} {dims}")

print("-" * 70)
print(f"\nAll three flawed reports should receive REVISE decisions.")
print(f"The specific dimensions flagged should match the injected flaws.")

## Part 8: Review Loop — Connecting Reviewer to Writer

The Review Agent's real power emerges when it's connected back to the Writing Agent in a **feedback loop**. This is the generate-critique-revise pattern from Notebook 07, applied at the system level.

```
    Writer                    Reviewer
      │                          │
      │   report v1              │
      ├─────────────────────────►│
      │                          │  evaluate across 5 dimensions
      │   REVISE + feedback      │
      │◄─────────────────────────┤
      │                          │
      │   report v2 (revised)    │
      ├─────────────────────────►│
      │                          │  re-evaluate
      │   PASS                   │
      │◄─────────────────────────┤
      │                          │
      ▼                          │
  Final Report                   │
```

In [ ]:
def revision_loop(question: str, sources: List[str],
                   max_rounds: int = 2, verbose: bool = True) -> dict:
    """Simulate the writer → reviewer → writer feedback loop.

    Uses the LLM as both writer and reviewer (different prompts).
    Returns dict with final report, revision history, and quality scores.
    """
    history = []
    reviewer = ReviewAgent(pass_threshold=7.0)

    # ── Step 1: Initial writing ──
    if verbose:
        print("REVISION LOOP")
        print("=" * 60)
        print(f"Question: {question}")
        print(f"Sources: {len(sources)}")
        print(f"Max revision rounds: {max_rounds}")
        print()

    src_text = "\n\n".join(f"[Source {i+1}] {s}" for i, s in enumerate(sources))
    writer_prompt = (
        f"Write a research report answering: {question}\n\n"
        f"Use these sources and cite them as [Source N]:\n{src_text}\n\n"
        f"Include: introduction, main findings organized by theme, and conclusion."
    )
    messages = [
        {"role": "system", "content":
            "You are a research report writer. Write a clear, well-cited report."},
        {"role": "user", "content": writer_prompt},
    ]
    report = generate(messages, max_new_tokens=800, temperature=0.7)

    if verbose:
        print(f"[Round 0: WRITE] Report generated ({len(report)} chars)")
        print(f"Preview: {report[:200]}...")
        print()

    for round_num in range(1, max_rounds + 1):
        if verbose:
            print(f"[Round {round_num}: REVIEW]")

        # Review the report
        result = reviewer.review(report, sources, question, verbose=verbose)
        history.append({
            "round": round_num,
            "report_length": len(report),
            "weighted_score": result.weighted_score,
            "decision": result.decision,
            "dimensions": {ds.dimension: ds.score for ds in result.dimension_scores},
            "feedback_count": len(result.feedback_items),
        })

        if result.decision == "PASS":
            if verbose:
                print(f"\n✓ PASSED at round {round_num}!")
            break

        # ── Revise ──
        if verbose:
            print(f"\n[Round {round_num}: REVISE] Sending {len(result.feedback_items)} feedback items to writer")
        feedback_text = reviewer.feedback_gen.format_for_writer(result.feedback_items)
        revise_messages = [
            {"role": "system", "content":
                "You are revising a research report based on reviewer feedback. "
                "Fix all issues while keeping what was good. Produce the COMPLETE revised report."},
            {"role": "user", "content":
                f"ORIGINAL QUESTION: {question}\n\n"
                f"CURRENT REPORT:\n{report}\n\n"
                f"REVIEWER FEEDBACK:\n{feedback_text}\n\n"
                f"SOURCES:\n{src_text}\n\n"
                f"Produce the complete revised report:"},
        ]
        report = generate(revise_messages, max_new_tokens=800, temperature=0.7)
        if verbose:
            print(f"  Revised report: {len(report)} chars")
            print()

    return {
        "final_report": report,
        "history": history,
        "total_rounds": len(history),
    }

print("✓ revision_loop defined")

In [ ]:
# Run the revision loop on one of our test cases
loop_result = revision_loop(
    question="Explain the Transformer architecture and its key innovations",
    sources=accuracy_sources,
    max_rounds=2,
    verbose=True,
)

print(f"\n{'=' * 60}")
print("FINAL REPORT:")
print("=" * 60)
print(loop_result["final_report"][:600] + "...")

In [ ]:
# Show quality scores across revision rounds
print("QUALITY PROGRESSION ACROSS ROUNDS")
print("=" * 60)

if loop_result["history"]:
    dims = list(loop_result["history"][0]["dimensions"].keys())
    header = f"{'Round':<8s} {'Score':<8s} {'Decision':<10s}"
    for d in dims:
        header += f" {d[:8]:<10s}"
    print(header)
    print("-" * len(header))

    for h in loop_result["history"]:
        row = f"{h['round']:<8d} {h['weighted_score']:<8.1f} {h['decision']:<10s}"
        for d in dims:
            row += f" {h['dimensions'].get(d, 'N/A')!s:<10s}"
        print(row)
else:
    print("No review rounds were needed.")

## Part 9: Comparison — No Review vs. Single-Pass vs. Multi-Dimensional Review

Now for the key experiment: does multi-dimensional review actually improve quality compared to simpler approaches?

| Approach | Description |
|----------|-------------|
| **No review** | Writer produces report, deliver immediately |
| **Single-pass review** | One LLM call: "rate this report 1-10 and suggest improvements" |
| **Multi-dimensional review** | Our 5-dimension ReviewAgent with specialized reviewers |

In [ ]:
def single_pass_review(report: str, sources: List[str], question: str) -> dict:
    """Simple single-pass review: one LLM call for overall assessment."""
    src_text = "\n".join(f"[{i+1}] {s[:100]}..." for i, s in enumerate(sources))
    messages = [
        {"role": "system", "content":
            "Rate this research report from 1 to 10. Identify any issues. "
            "Respond with JSON: {\"score\": N, \"issues\": [...], \"summary\": \"...\"}"},
        {"role": "user", "content":
            f"Question: {question}\nSources: {src_text}\nReport: {report}"},
    ]
    raw = generate(messages, max_new_tokens=300, temperature=0.3, do_sample=True)
    try:
        match = re.search(r'\{[\s\S]*\}', raw)
        if match:
            data = json.loads(match.group())
            return {"score": data.get("score", 5), "issues": data.get("issues", []),
                    "summary": data.get("summary", "")}
    except (json.JSONDecodeError, ValueError):
        pass
    return {"score": 5, "issues": ["Parse error"], "summary": raw[:200]}


# Run the comparison on the factual-errors report
print("COMPARISON: Review Approaches on Factually Flawed Report")
print("=" * 60)

# Approach 1: No review
print("\n1. NO REVIEW")
print(f"   Score: N/A (report delivered as-is)")
print(f"   Issues caught: 0")
print(f"   The factual errors would reach the user.")

# Approach 2: Single-pass review
print("\n2. SINGLE-PASS REVIEW")
single_result = single_pass_review(
    flawed_report_accuracy, accuracy_sources,
    "Explain the Transformer architecture"
)
print(f"   Score: {single_result['score']}/10")
print(f"   Issues caught: {len(single_result['issues'])}")
for issue in single_result["issues"][:3]:
    print(f"     • {issue}")

# Approach 3: Multi-dimensional review
print("\n3. MULTI-DIMENSIONAL REVIEW (our ReviewAgent)")
print(f"   Weighted Score: {result_accuracy.weighted_score:.1f}/10")
print(f"   Issues caught: {len(result_accuracy.feedback_items)}")
print(f"   Dimensions flagged: {', '.join(result_accuracy.dimensions_to_improve)}")
for ds in result_accuracy.dimension_scores:
    print(f"     {ds.dimension}: {ds.score}/10")

print(f"\n{'=' * 60}")
print("TAKEAWAY: Multi-dimensional review catches more issues because")
print("each dimension gets focused attention from a specialized reviewer.")

In [ ]:
# Build comparison table
print("\n" + "=" * 70)
print("COMPARISON TABLE: Review Approach Quality")
print("=" * 70)
print(f"{'Metric':<30s} {'No Review':<15s} {'Single-Pass':<15s} {'Multi-Dim':<15s}")
print("-" * 70)

metrics = [
    ("Issues detected", "0", str(len(single_result["issues"])),
     str(len(result_accuracy.feedback_items))),
    ("Dimensions evaluated", "0", "1 (overall)", "5 (specialized)"),
    ("Actionable feedback items", "0", "maybe",
     str(len(result_accuracy.feedback_items))),
    ("Severity classification", "N/A", "N/A", "✓"),
    ("Location in report", "N/A", "N/A", "✓"),
    ("Self-consistency check", "N/A", "N/A", "✓ (reflection)"),
    ("LLM calls", "0", "1", str(len(RUBRIC_DIMENSIONS) + 1)),
    ("Pass/revise decision", "N/A", "manual", "automatic"),
]

for name, *vals in metrics:
    print(f"{name:<30s} {vals[0]:<15s} {vals[1]:<15s} {vals[2]:<15s}")
print("-" * 70)
print("Multi-dimensional review costs more LLM calls but catches more issues,")
print("provides structured feedback, and enables automatic revision loops.")

## Part 10: Preview — Connecting to the Full System (Notebook 37)

In Notebook 37 we'll connect the Review Agent into the complete Castalia Scholar pipeline:

```
Orchestrator
  ├── PlannerAgent    → sub-questions
  ├── RetrieverAgent  → relevant sources
  ├── AnalyzerAgent   → synthesis + insights
  ├── WriterAgent     → research report
  └── ReviewerAgent   → quality gate
        │
        ├── PASS → deliver to user
        └── REVISE → back to WriterAgent (max 2 rounds)
```

The Review Agent is what turns a **linear pipeline** into a **quality-assured system**. Without it, you have a pipeline. With it, you have a system that **self-corrects**.

## Summary & Key Takeaways

### What We Built

| Component | Purpose | Key Technique |
|-----------|---------|---------------|
| **ReviewRubric** | Defines evaluation dimensions with weights | Structured scoring criteria |
| **DimensionReviewer** | Evaluates one quality dimension deeply | Focused prompts per dimension |
| **FeedbackGenerator** | Converts scores to actionable feedback | Severity mapping + localization |
| **ReviewDecision** | Pass/revise logic with thresholds | Weighted scoring + per-dimension minimums |
| **ReviewAgent** | Complete review pipeline | Combines all + reflection for consistency |

### Key Insights

1. **Multi-dimensional review > single-pass review** — specialized reviewers catch more issues because each focuses on one aspect
2. **Structured feedback > "make it better"** — the writer needs to know what, where, and how to fix
3. **Thresholds matter** — too low and bad reports pass; too high and the system never converges
4. **Reflection improves reviews** — checking review consistency catches contradictory scores
5. **The review loop is the quality multiplier** — each revision round measurably improves the report

### Design Patterns Used

- **Reflection** (Notebook 07) — reviewer reflects on its own assessment
- **Structured output** — JSON parsing for dimension scores
- **Separation of concerns** — each dimension gets a specialized reviewer
- **Feedback loop** — reviewer output feeds writer revision

### Next Steps

→ **Notebook 37**: Full system orchestration — all agents connected in a complete pipeline with the review loop integrated.